In [0]:

# 1. SETUP DE AMBIENTE E SCHEMAS
# DECISÃO: repetimos a definição de catalog/schema aqui em vez de importar
# do notebook Bronze. Cada notebook do Job roda em um contexto de execução
# isolado no Databricks Workflow — não há import direto entre notebooks sem
# usar %run ou um módulo compartilhado. Preferimos essa pequena duplicação a
# acoplar os notebooks via %run, que tornaria mais difícil rodar cada etapa
# isoladamente para debug

catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Camada Bronze origem: {bronze_schema}")
print(f"Camada Silver destino: {silver_schema}")

Camada Bronze origem: workspace.bronze
Camada Silver destino: workspace.silver


In [0]:

# 2. SILVER: TB_INFO_FILMES (ORIGEM: BRONZE.TB_MOVIES_INFO)

# DECISÃO 1 (dedup): Window por `id` ordenada por `ingestion_datetime DESC` +
# `row_number()==1`, em vez de `dropDuplicates(["id"])`. O motivo é que
# `dropDuplicates` não garante qual linha sobrevive quando há duplicatas —
# a regra de negócio exige explicitamente "a versão mais recente", então
# precisávamos de controle determinístico sobre o critério de desempate

# DECISÃO 2 (status): normalizamos (upper + trim + remoção de hífens) ANTES
# de comparar com a lista de status válidos, e não depois. Assim, variações
# como "in-production", "IN_PRODUCTION " ou "In Production" caem todas no
# mesmo `WHEN`, em vez de cada variação exigir sua própria cláusula — o que
# seria frágil e cresceria a cada novo ruído encontrado na origem

# DECISÃO 3 (datas): `coalesce` de múltiplos `try_to_date()` tenta os
# formatos do mais específico (ISO) ao mais ambíguo, e para no primeiro que
# funcionar. Isso resolve o pedido de "testar diferentes padrões de forma
# robusta" sem lançar exceção em modo ANSI — mas é importante registrar que
# formatos ambíguos como "01/02/2020" podem ser mal interpretados (dia/mês
# trocados) se dois padrões testados aceitarem a mesma string com resultados
# diferentes; a ordem da lista de coalesce implicitamente prioriza um deles

# DECISÃO 4 (duração): extraímos apenas os dígitos iniciais via regex antes
# do `try_cast`, em vez de só fazer `try_cast(runtime AS INT)` direto. Isso
# recupera valores como "120min" ou "120 min" que um cast puro descartaria
# como NULL — outra forma de reduzir perda de dado por ruído de formatação

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

# 1. Deduplicação por filme mantendo o registro mais recente
janela_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_info_dedup = (
    df_bronze_info
    .withColumn("row_num", F.row_number().over(janela_dedup))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# 2. Normalização e Tradução do Status
status_limpo = F.upper(F.trim(F.regexp_replace(F.col("status"), r"[-_]+", " ")))

status_traduzido = (
    F.when(status_limpo == "RELEASED", "Lançado")
    .when(status_limpo == "POST PRODUCTION", "Pós-Produção")
    .when(status_limpo == "IN PRODUCTION", "Em Produção")
    .when(status_limpo == "PLANNED", "Planejado")
    .when(status_limpo == "RUMORED", "Rumores")
    .when(status_limpo == "CANCELED", "Cancelado")
    .otherwise("Não Informado")
)

# 3. Conversão de Data Multi-Formato Robusta com try_to_date (Tolerante a ANSI)
data_convertida = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
    F.expr("try_to_date(release_date, 'MM/dd/yyyy')"),
    F.expr("try_to_date(release_date, 'yyyy/MM/dd')"),
    F.expr("try_to_date(release_date, 'd/M/yyyy')")
)

# 4. Extração e conversão segura da duração (protegida contra string vazia)
# Extrai dígitos iniciais e utiliza try_cast para converter com segurança para INT
duracao_segura = F.expr("try_cast(regexp_extract(runtime, '^(\\\\d+)', 1) AS INT)")

# 5. Seleção com Tipagem Estrita e Nomes Canónicos em Português
df_silver_info_filmes = (
    df_info_dedup
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.col("title").cast("string").alias("titulo"),
        F.col("original_title").cast("string").alias("titulo_original"),
        data_convertida.alias("data_lancamento"),
        F.year(data_convertida).cast("int").alias("ano_lancamento"),
        duracao_segura.alias("duracao_minutos"),
        F.col("original_language").cast("string").alias("idioma_original"),
        status_traduzido.alias("status_filme"),
        F.col("overview").cast("string").alias("sinopse"),
        F.col("tagline").cast("string").alias("frase_divulgacao"),
        F.current_timestamp().alias("silver_ingestion_datetime")
    )
)

# 6. Persistência em Delta Lake na Camada Silver
(
    df_silver_info_filmes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_info_filmes")
)

print(f" Tabela {silver_schema}.tb_info_filmes gravada com sucesso ({df_silver_info_filmes.count()} registros).")

 Tabela workspace.silver.tb_info_filmes gravada com sucesso (97611 registros).


In [0]:

# 3. SILVER: TB_COTACAO_DOLAR (ORIGEM: BRONZE.TB_COTACAO_DOLAR)

# DECISÃO: construímos um calendário contínuo (`sequence` de datas) e fazemos
# LEFT JOIN com a cotação real, em vez de só usar as datas que a API
# retornou. Isso é o que torna o forward-fill possível: sem essa "grade" de
# datas, não haveria nenhum dia "vazio" para preencher — a tabela teria
# apenas os dias em que a API respondeu

# 1. Normaliza as cotações da Bronze para grão diário (data pura)
df_cotacao_bronze = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .withColumn("data_referencia", F.to_date(F.col("dataHoraCotacao")))
    .groupBy("data_referencia")
    .agg(F.last("cotacaoCompra").alias("cotacao_usd_brl"))
)

# 2. Identifica o intervalo de datas necessário (ou usa uma janela padrão ampla)
min_max_datas = df_cotacao_bronze.select(
    F.min("data_referencia").alias("min_dt"),
    F.max("data_referencia").alias("max_dt")
).collect()[0]

min_data = min_max_datas["min_dt"]
max_data = min_max_datas["max_dt"]

# Caso a Bronze só contenha a janela de 7 dias, garantimos um calendário diário
df_calendario = spark.sql(f"""
    SELECT explode(sequence(DATE '{min_data}', DATE '{max_data}', INTERVAL 1 DAY)) AS data_referencia
""")

# 3. Left join do calendário contínuo com as cotações existentes
df_serie_temporal = (
    df_calendario.join(df_cotacao_bronze, on="data_referencia", how="left")
)

# 4. Forward Fill: preenche nulos com a última cotação não-nula anterior
janela_ffill = (
    Window.orderBy("data_referencia")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_silver_cotacao = (
    df_serie_temporal
    .withColumn(
        "cotacao_usd_brl",
        F.last("cotacao_usd_brl", ignorenulls=True).over(janela_ffill)
    )
    .select(
        F.col("data_referencia").cast("date"),
        F.col("cotacao_usd_brl").cast("decimal(10,4)").alias("taxa_cotacao"),
        F.current_timestamp().alias("silver_ingestion_datetime")
    )
)

(
    df_silver_cotacao.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")
)

print(f"✔ Tabela {silver_schema}.tb_cotacao_dolar gravada ({df_silver_cotacao.count()} dias contínuos).")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✔ Tabela workspace.silver.tb_cotacao_dolar gravada (9 dias contínuos).


In [0]:

# 4. SILVER: TB_FINANCEIRO_FILMES (ORIGEM: BRONZE.TB_MOVIES_FINANCIALS)

# DECISÃO 1 (mapeamento origem→destino): o documeto do projeto descreve o mapeamento
# de forma um pouco ambígua ("orcamento_usd -> orcamento_brl" usando o nome
# de destino como se fosse origem). Interpretamos como: budget/revenue
# (brutos, limpos) viram orcamento_usd/receita_usd, e estes SIM são a origem
# para orcamento_brl/receita_brl via multiplicação pela cotação.

# DECISÃO 2 (limpeza): usamos uma função nomeada (`limpar_valor_financeiro`)
# reaplicada em budget e revenue, em vez de duplicar a lógica duas vezes.
# Evita que uma correção futura na regra de limpeza (ex.: um novo texto de
# "ausência de dado" aparecer na origem) precise ser replicada em dois
# lugares e um deles seja esquecido

# DECISÃO 3 (valores ≤ 0 → NULL): tratamos orçamento/receita zerados ou
# negativos como ausência de dado, não como "filme sem receita reportada".
# Um orçamento de $0 quase certamente é dado faltante na origem (TMDB/IMDb
# raramente têm filmes de orçamento zero real), então nulificar evita
# distorcer médias e somas para baixo artificialmente

# DECISÃO 4 (taxa de câmbio): fazemos left join com `tb_cotacao_dolar` pela
# data de lançamento do filme, com fallback para a última cotação disponível
# quando não há match

# Ponto de atenção: dado o tamanho da
# janela da API (célula anterior), o fallback é acionado para a esmagadora
# maioria dos filmes — ou seja, na prática quase toda a base usa uma taxa de
# câmbio "atual" aplicada retroativamente a filmes de qualquer década. Isso é
# uma simplificação que pode  distorcer
# comparações de lucro/margem entre filmes de períodos muito diferentes
# (ex.: um filme de 1995 convertido pela cotação de 2026 não reflete o valor
# real em BRL da época)

# DECISÃO 5 (margem de lucro): protegemos contra divisão por zero checando
# `receita_usd > 0` antes de dividir, em vez de usar `try_divide` puro,
# porque queremos NULL tanto para receita nula quanto para receita zero
# (que também tornaria a margem sem sentido de negócio, mesmo sem erro
# técnico de divisão)

df_bronze_financials = spark.table(f"{bronze_schema}.tb_movies_financials")
df_info = spark.table(f"{silver_schema}.tb_info_filmes").select("id_filme", "data_lancamento")
df_cotacao = spark.table(f"{silver_schema}.tb_cotacao_dolar")

# Taxa de cotação padrão (última cotação disponível) como fallback seguro
ultima_cotacao_valor = (
    df_cotacao.orderBy(F.col("data_referencia").desc())
    .select("taxa_cotacao")
    .first()[0]
)

def limpar_valor_financeiro(coluna):
    """
    Remove símbolos de dólar, vírgulas de milhar e espaços, tratando
    textos representativos de ausência de dados como NULL.
    """
    col_str = F.trim(coluna)
    eh_ausente = (
        (col_str.isNull()) |
        (col_str == "") |
        (F.upper(col_str).isin("UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "NULL", "NONE"))
    )
    # Remove qualquer caractere que não seja número ou ponto decimal
    valor_limpo = F.regexp_replace(col_str, r"[^0-9.]", "")
    
    # Cast para Decimal e anula valores <= 0
    valor_decimal = F.when(valor_limpo == "", None).otherwise(valor_limpo).cast("decimal(18,2)")
    return F.when(eh_ausente | (valor_decimal <= 0), None).otherwise(valor_decimal)

# Deduplica financeiro caso haja IDs duplicados
janela_fin = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_fin_dedup = (
    df_bronze_financials
    .withColumn("rn", F.row_number().over(janela_fin))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_fin_limpo = (
    df_fin_dedup
    .withColumn("id_filme", F.col("id").cast("string"))
    .withColumn("orcamento_usd", limpar_valor_financeiro(F.col("budget")))
    .withColumn("receita_usd", limpar_valor_financeiro(F.col("revenue")))
    # Cruzamento com info para obter data_lancamento
    .join(df_info, on="id_filme", how="left")
    # Cruzamento com cotacao do dólar na data do filme
    .join(df_cotacao, df_info.data_lancamento == df_cotacao.data_referencia, how="left")
    # Fallback para a cotação mais recente se a data não cruzar
    .withColumn("taxa_aplicada", F.coalesce(F.col("taxa_cotacao"), F.lit(ultima_cotacao_valor)))
)

# Cálculos em BRL, Lucros e Margens de Lucro
df_silver_financials = (
    df_fin_limpo
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("taxa_aplicada")).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.col("taxa_aplicada")).cast("decimal(18,2)"))
    .withColumn(
        "lucro_usd",
        F.when(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull(),
               F.col("receita_usd") - F.col("orcamento_usd")).otherwise(None)
    )
    .withColumn(
        "lucro_brl",
        F.when(F.col("orcamento_brl").isNotNull() & F.col("receita_brl").isNotNull(),
               F.col("receita_brl") - F.col("orcamento_brl")).otherwise(None)
    )
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            (F.col("receita_usd").isNotNull()) & (F.col("receita_usd") > 0) & (F.col("lucro_usd").isNotNull()),
            F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2)
        ).otherwise(None)
    )
    .select(
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_percentual",
        F.current_timestamp().alias("silver_ingestion_datetime")
    )
)

(
    df_silver_financials.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")
)

print(f"✔ Tabela {silver_schema}.tb_financeiro_filmes gravada ({df_silver_financials.count()} registros).")

✔ Tabela workspace.silver.tb_financeiro_filmes gravada (99006 registros).


In [0]:

# 5. SILVER: TB_METRICAS_ENGAJAMENTO (ORIGEM: BRONZE.TB_MOVIES_METRICS)

# DECISÃO: REFINO DE DATA QUALITY,
# - Blindagem contra Column Shift severo:
#   Textos e resíduos alfabéticos são convertidos para NULL via try_cast.
#    Detecção de Anos Deslocados na Popularidade em versões anteriores do código: filmes onde a popularidade
#    veio exatamente com o valor de um ano (ex: 2020.0, 2019.0, 1969.0) devido
#    ao deslocamento de colunas têm sua popularidade tratada como NULL, por isso a "Blindagem"
#   3. Validação estrita de notas (0 a 10) e contagem de votos (>= 0)


from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

janela_met = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_met_dedup = (
    df_bronze_metrics
    .withColumn("rn", F.row_number().over(janela_met))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Popularidade: limpa pontuação, faz try_cast e remove valores que são anos inteiros (ex: 1900 a 2030 redondos)
expr_popularidade = """
    CASE 
        WHEN try_cast(regexp_replace(trim(popularity), ',', '.') AS DOUBLE) >= 0.0 
             -- Filtra Column Shift onde o ano de 4 dígitos foi parar na popularidade
             AND NOT (
                 try_cast(regexp_replace(trim(popularity), ',', '.') AS DOUBLE) BETWEEN 1880.0 AND 2035.0
                 AND try_cast(regexp_replace(trim(popularity), ',', '.') AS DOUBLE) = floor(try_cast(regexp_replace(trim(popularity), ',', '.') AS DOUBLE))
             )
        THEN try_cast(regexp_replace(trim(popularity), ',', '.') AS DOUBLE)
        ELSE NULL 
    END
"""

expr_nota_tmdb = """
    CASE 
        WHEN try_cast(regexp_replace(trim(vote_average), ',', '.') AS DOUBLE) BETWEEN 0.0 AND 10.0 
        THEN try_cast(regexp_replace(trim(vote_average), ',', '.') AS DOUBLE)
        ELSE NULL 
    END
"""

expr_nota_imdb = """
    CASE 
        WHEN try_cast(regexp_replace(trim(averageRating), ',', '.') AS DOUBLE) BETWEEN 0.0 AND 10.0 
        THEN try_cast(regexp_replace(trim(averageRating), ',', '.') AS DOUBLE)
        ELSE NULL 
    END
"""

expr_votos_tmdb = """
    CASE 
        WHEN try_cast(trim(vote_count) AS INT) >= 0 
        THEN try_cast(trim(vote_count) AS INT)
        ELSE NULL 
    END
"""

expr_votos_imdb = """
    CASE 
        WHEN try_cast(trim(numVotes) AS INT) >= 0 
        THEN try_cast(trim(numVotes) AS INT)
        ELSE NULL 
    END
"""

df_silver_metrics = (
    df_met_dedup
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.expr(expr_popularidade).alias("popularidade"),
        F.expr(expr_nota_tmdb).alias("nota_media_tmdb"),
        F.expr(expr_votos_tmdb).alias("qtd_votos_tmdb"),
        F.expr(expr_nota_imdb).alias("nota_media_imdb"),
        F.expr(expr_votos_imdb).alias("qtd_votos_imdb"),
        F.current_timestamp().alias("silver_ingestion_datetime")
    )
)

(
    df_silver_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")
)

print(f"✔ Tabela {silver_schema}.tb_metricas_engajamento regravada com saneamento de popularidade.")

✔ Tabela workspace.silver.tb_metricas_engajamento regravada com saneamento de popularidade.


In [0]:

# 6. SILVER: TB_AVALIACOES_USUARIOS (ORIGEM: BRONZE.TB_MOVIES_REVIEWS)

# DECISÃO 1 (ordem das operações): validamos a nota (fora de 0–10 vira NULL)
# ANTES de aplicar `dropDuplicates`. Isso é uma decisão consciente, mas com
# uma consequência que vale documentar: dois registros originais inválidos
# e diferentes (ex. nota "15" e nota "-3") podem colapsar em um único
# registro após ambos virarem NULL, mesmo não sendo duplicatas reais na
# origem. Optamos por manter assim porque a regra pede deduplicação sobre os
# campos finais da Silver (já tratados), não sobre o dado bruto.

# DECISÃO 2 (comentário vazio → "Sem comentário"): preenchemos com texto
# padronizado em vez de deixar NULL, porque este campo alimenta leitura
# humana/relatórios de BI onde "NULL" apareceria como célula vazia confusa;
# um texto explícito também facilita filtros posteriores
# (`comentario_usuario != 'Sem comentário'`) sem precisar lidar com
# `isNull()` em toda consulta futura


df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

# 1. Higienização de notas e comentários
nota_convertida = F.regexp_replace(F.trim(F.col("nota")), ",", ".").cast("double")
nota_validada = F.when((nota_convertida >= 0.0) & (nota_convertida <= 10.0), nota_convertida).otherwise(None)

comentario_limpo = F.trim(F.col("comentario"))
comentario_padronizado = (
    F.when(comentario_limpo.isNull() | (comentario_limpo == ""), "Sem comentário")
    .otherwise(comentario_limpo)
)

df_silver_reviews = (
    df_bronze_reviews
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.col("nome").cast("string").alias("nome_usuario"),
        nota_validada.alias("nota_usuario"),
        comentario_padronizado.alias("comentario_usuario")
    )
    # Deduplicação integral obrigatória
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
    .withColumn("silver_ingestion_datetime", F.current_timestamp())
)

(
    df_silver_reviews.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")
)

print(f"✔ Tabela {silver_schema}.tb_avaliacoes_usuarios gravada ({df_silver_reviews.count()} registros).")

✔ Tabela workspace.silver.tb_avaliacoes_usuarios gravada (32412 registros).


In [0]:

# 7. SILVER: TB_GENEROS (ORIGEM: BRONZE.TB_CREDITS_AND_TAGS)

# DECISÃO 1 (normalizar `;` para `,` antes do split): a origem mistura os
# dois delimitadores. Convertê-los para um único caractere antes do split
# evita ter que fazer um regex de split mais complexo (`split(col, "[,;]")`)
# — mas atenção: um regex de split direto seria funcionalmente equivalente e
# mais direto; a troca prévia foi só uma preferência de legibilidade.

# DECISÃO 2 (filtro anti-ruído): descartamos strings puramente numéricas e
# strings com 1 caractere ou menos, porque o Column Shift mencionado no
# documento do projeto espalha fragmentos numéricos/vazios pela coluna de gêneros. Esse
# filtro é heurístico — pode descartar um gênero legítimo muito curto (não
# há nenhum caso conhecido no domínio de cinema) e pode deixar passar ruído
# textual de 2+ caracteres que não é um gênero real. Vale mencionar essa
# limitação como trade-off aceito

#Ponto de atenção: `initcap()` capitaliza cada palavra, o que pode descaracterizar siglas: "TV Movie"
# vira "Tv Movie". Se houver a necessidade de correção para o futuro do projetor, vale uma lista de exceções conhecidas (TV,
# UK, USA etc.) aplicada depois do `initcap()`

df_credits_raw = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Normaliza múltiplos delimitadores (, ; |) para vírgula antes do split
generos_padronizados = F.regexp_replace(F.col("genres"), r"[;|]", ",")

df_generos_exploded = (
    df_credits_raw
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(F.split(generos_padronizados, ",")).alias("nome_genero_raw")
    )
)

# Lista canônica dos gêneros oficiais do cinema (TMDB / IMDb)
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", 
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery", 
    "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]

# Normaliza caixa e trata sigla TV Movie
nome_genero_tratado = (
    F.when(F.lower(F.trim(F.col("nome_genero_raw"))).isin("tv movie", "tvmovie"), "TV Movie")
     .otherwise(F.initcap(F.trim(F.col("nome_genero_raw"))))
)

df_silver_generos = (
    df_generos_exploded
    .withColumn("nome_genero", nome_genero_tratado)
    # Filtra estritamente pelo domínio legítimo de gêneros cinematográficos
    .filter(F.col("nome_genero").isin(generos_validos))
    .select("id_filme", "nome_genero")
    .dropDuplicates(["id_filme", "nome_genero"])
    .withColumn("silver_ingestion_datetime", F.current_timestamp())
)

(
    df_silver_generos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_generos")
)

print(f"✔ Tabela {silver_schema}.tb_generos saneada com sucesso ({df_silver_generos.count()} registros).")

✔ Tabela workspace.silver.tb_generos saneada com sucesso (141941 registros).


In [0]:

# 8. SILVER: TB_PESSOAS_EMPRESAS (ORIGEM: BRONZE.TB_CREDITS_AND_TAGS)

# DECISÃO: encapsulamos a lógica de explode+limpeza numa função
# (`desmembrar_entidades`) parametrizada por coluna de origem e tipo de
# entidade, em vez de escrever o mesmo bloco 4 vezes (cast, directors,
# writers, production_companies). Isso segue diretamente a recomendação do
# projeto de "dimensão unificada" — a função é a materialização em código
# dessa ideia: mesma regra de limpeza, mesmo filtro anti-ruído, aplicados
# uniformemente aos 4 tipos, evitando que uma correção futura precise ser
# replicada em 4 lugares.
# Mesma ressalva do `initcap()` da célula anterior se aplica aqui,
# potencialmente afetando siglas de produtoras (ex. "HBO" → "Hbo")

def desmembrar_entidades(coluna_origem: str, tipo_entidade: str):
    """
    Aplica normalização de separadores, explode itens, limpa textos
    e anexa a respectiva tipificação da entidade.
    """
    col_norm = F.regexp_replace(F.col(coluna_origem), ";", ",")
    return (
        df_credits_raw
        .select(
            F.col("id").cast("string").alias("id_filme"),
            F.explode(F.split(col_norm, ",")).alias("nome_entidade_raw"),
            F.lit(tipo_entidade).alias("tipo_entidade")
        )
        .withColumn("nome_entidade", F.initcap(F.trim(F.col("nome_entidade_raw"))))
        .filter(
            (F.col("nome_entidade").isNotNull()) &
            (F.col("nome_entidade") != "") &
            (~F.col("nome_entidade").rlike(r"^\d+$")) &
            (F.length(F.col("nome_entidade")) > 1)
        )
        .select("id_filme", "nome_entidade", "tipo_entidade")
    )

df_atores = desmembrar_entidades("cast", "Ator")
df_diretores = desmembrar_entidades("directors", "Diretor")
df_roteiristas = desmembrar_entidades("writers", "Roteirista")
df_produtoras = desmembrar_entidades("production_companies", "Produtora")

# União de todas as entidades
df_silver_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
    .withColumn("silver_ingestion_datetime", F.current_timestamp())
)

(
    df_silver_pessoas_empresas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")
)

print(f" Tabela {silver_schema}.tb_pessoas_empresas gravada ({df_silver_pessoas_empresas.count()} registros).")

 Tabela workspace.silver.tb_pessoas_empresas gravada (915307 registros).


In [0]:

# 9. VALIDAÇÃO E AUDITORIA DA CAMADA SILVER
# DECISÃO: listamos as 7 tabelas esperadas explicitamente (hardcoded), em
# vez de descobrir dinamicamente via `SHOW TABLES IN silver`. Isso é
# intencional: queremos que o smoke test FALHE ruidosamente se uma tabela
# esperada não existir (ex.: célula anterior quebrou silenciosamente), em
# vez de simplesmente reportar "o que existe" sem confrontar com "o que
# deveria existir"

tabelas_silver_esperadas = [
    "tb_info_filmes",
    "tb_cotacao_dolar",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas"
]

print("=" * 65)
print(f"AUDITORIA DE TABELAS CRIADAS NO SCHEMA '{silver_schema}':")
print("=" * 65)

for tabela in tabelas_silver_esperadas:
    nome_full = f"{silver_schema}.{tabela}"
    try:
        contagem = spark.table(nome_full).count()
        print(f" {nome_full:<40} | Linhas: {contagem}")
    except Exception as e:
        print(f" Erro na tabela {nome_full}: {e}")

print("=" * 65)
display(spark.table(f"{silver_schema}.tb_info_filmes").limit(5))

AUDITORIA DE TABELAS CRIADAS NO SCHEMA 'workspace.silver':
 workspace.silver.tb_info_filmes          | Linhas: 97611
 workspace.silver.tb_cotacao_dolar        | Linhas: 9
 workspace.silver.tb_financeiro_filmes    | Linhas: 99006
 workspace.silver.tb_metricas_engajamento | Linhas: 95115
 workspace.silver.tb_avaliacoes_usuarios  | Linhas: 32412
 workspace.silver.tb_generos              | Linhas: 141941
 workspace.silver.tb_pessoas_empresas     | Linhas: 915307


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,silver_ingestion_datetime
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2026-09-20T18:00:36.272Z
1000014,On va manquer !,On va manquer !,2018-05-15,2018,0,fr,Lançado,null,null,2026-09-20T18:00:36.272Z
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null,null,2026-09-20T18:00:36.272Z
1000088,Monsieur le Maire,Monsieur le Maire,2023-11-01,2023,0,fr,Pós-Produção,null,null,2026-09-20T18:00:36.272Z
1000091,Amy Miller: Ham Mouth,Amy Miller: Ham Mouth,2022-03-24,2022,35,en,Lançado,"Amy Miller reflects on her recent breakup, shares her love of baths and reveals the 40-year-old shit she does.",null,2026-09-20T18:00:36.272Z
